# UniPercept — Python quickstart

`unipercept` is a Cython extension over the UniPercept C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install lituus-unipercept
```

CI executes this notebook against the wheel the release actually publishes, so
an output below that stops matching fails the build.

## The API

`decode` returns an `Image`; aHash, dHash and pHash return 64-bit integers, blockhash returns bytes, and Hamming distance counts the differing bits. `BkTree` performs exact radius queries over stored hashes.

In [1]:
import unipercept

unipercept.version(), unipercept.__version__, unipercept.abi_version()

('1.0.0', '1.0.0', 2)

## Decode and hash a small image

A 2x2 RGB PPM (uncompressed, so the example needs no encoder):

In [2]:
PPM = b"P6\n2 2\n255\n" + bytes([0xFF,0,0, 0,0xFF,0, 0,0,0xFF, 0xFF,0xFF,0xFF])
img = unipercept.decode(PPM)
img.width, img.height, img.channels

(2, 2, 3)

In [3]:
a, d, p = img.ahash(), img.dhash(), img.phash()
a, d, p

(17361641481138401520, 1157442765409226768, 15996635242756536029)

In [4]:
unipercept.hamming(a, a), unipercept.hamming(a, d)

(0, 24)

In [5]:
bh = img.blockhash(16)
len(bh), bh[:8].hex()

(32, '00ff00ff00ff00ff')

## Grayscale kernels and a similarity index

In [6]:
gray = unipercept.to_grayscale(PPM[-12:], 2, 2, 3)
gray.pixels, gray.phash() == p

(b'L\x95\x1d\xff', True)

In [7]:
tree = unipercept.BkTree()
tree.insert(10, p)
tree.insert(11, d)
len(tree), sorted(tree.query(p, radius=64))

(2, [(10, 0), (11, 44)])